1. How do the feature types in the UCI Heart Disease dataset (combining continuous physiological telemetry like chol and thalach with categorical/ordinal indicators like cp and thal) differ from previous datasets, and why is proper scaling critical for Support Vector Machines compared to Decision Trees?

The UCI Heart Disease dataset mixes continuous measurements with categorical clinical indicators that are stored as numbers but are really labels. This is different from other datasets due to the fact that I needed to differentiate between what should be a category and what needed to be an object. Because every column is stored as numeric, I had to separate the features by clinical meaning, impute each group differently (median for continuous, most frequent for categorical), and one-hot encode the categorical group so the models do not read a false order into the codes. Scaling is critical for the RBF kernel SVC because it relies on distances between patients, so a feature with a large range like chol (mean: 246.7, std 51.8) would dominate one with a small range like oldpeak (mean 1.0, std 1.2) regardless of clinical importance, and a single gamma is applied to every dimension. Standardizing puts all continuous features on equal footing. A Decision Tree does not need scaling because it splits on one feature at a time using thresholds that depend only on the ordering of values, so any monotonic rescaling gives the same splits.

2. Explain how hidden missing values (such as ? symbols) impact raw medical datasets, and justify your choice of imputation strategies for skewed numerical vitals versus categorical diagnostic fields.

In raw medical datasets, missing values are often hidden behind placeholders like “?”, which pandas reads as text instead of as missing, so isnull() reports zero null values and gives a false sense of completeness. This can also turn numeric columns into text columns, and if the placeholders are left alone they either crash models like SVC, which cannot accept NaN, or get treated as a real category and quietly distort the results. Dropping incomplete rows is also a poor fix on a dataset of only about 300 patients, because it throws away valuable records and the missingness may not be random. I checked for “?” explicitly and found zero results due to the loader already converting them to NaN, and the remaining missing values were in ca and thal . For the skewed numerical vitals I used median imputation because the mean is pulled toward outliers and skewed tails. For the categorical fields, I used most frequent imputation because a mean could produce values that match no real category. Both imputers sit inside the pipeline, so they are fit on the training data only, which prevents data leakage into the test set.

3. Contrast how a Support Vector Classifier (using an RBF kernel) constructs decision boundaries versus how a Decision Tree recursively partitions feature space. What are the interpretability tradeoffs between the two models?

A Decision Tree recursively partitions the feature space by repeatedly asking yes/no questions about a single feature. SVC treats each patient as a data point and finds the boundary that separates the two classes with the widest possible margin, and the kernel lets that boundary curve around clusters of patients by considering multiple features at once. The support vectors, which are the points closest to the boundary or within the margin, define its shape. Decision Trees are easier to interpret because each prediction follows a clear set of rules that a doctor can verify, but their splits struggle with complex relationships, and my depth-2 tree left two leaves evenly mixed. The SVC captures nonlinear patterns more effectively but is harder to interpret, since its predictions cannot be explained through a simple sequence of rules. 

4. Discuss how tuning hyperparameters such as regularization parameter C and kernel coefficient gamma in an SVC impacts the trade-off between bias, variance, and model generalization on unseen medical test data.

The C and gamma parameters control how complex the SVM decision boundary becomes, which sets the balance between bias and variance. A smaller C allows more misclassifications, creating a smoother boundary with higher bias and lower variance and a higher risk of underfitting, while a larger C prioritizes correctly classifying the training data, giving lower bias but higher variance and a higher risk of overfitting. Similarly, a smaller gamma lets each data point influence a larger area, producing a smooth boundary (high bias, low variance), while a larger gamma makes influence very local, producing a complex boundary that focuses on individual patients (low bias, high variance). Finding the right balance is especially important for medical data because the model needs to generalize to new patients rather than memorize the training data, so I used 5-fold cross-validation with ROC-AUC scoring to choose the values. My grid search selected C = 10 and gamma = 0.001, achieving a best cross-validation AUC of 0.8947. The tuned model achieved a test AUC of 0.9643, the same as the default SVC, indicating that tuning did not improve performance on the test set.

5. Inspecting your Decision Tree feature importances or model coefficients, which clinical indicators (e.g., maximum heart rate, chest pain type, ST depression) exhibit the strongest predictive power for heart disease, and what preventative screening insights do these results provide for cardiologists?
 
Based on my Decision Tree, the strongest predictors of heart disease were thal, cp, and ca. The tree first split on thal, suggesting it was the most useful feature for separating patients with and without heart disease. Chest pain type and the number of major vessels were then used to further classify patients. For example, one group identified using thal and chest pain type contained 64 patients with heart disease out of 72, while another group identified using thal and ca contained 86 patients without heart disease out of 97. These results suggest that doctors should pay particular attention to thalassemia test results, chest pain symptoms, and coronary vessel assessments when evaluating a patient's risk of heart disease. Combining these indicators could help identify higher risk patients who may benefit from additional screening or preventative care.